# 08 — Continue training Dueling Double DQN

Starting from the frozen `dueling_hunt_center_04` checkpoint, this experiment changes only the opponent distribution while preserving the network, features, rewards, learning rate, exploration schedule, and target-network logic. Training runs for 1,000 rounds, freezes a checkpoint every 200 rounds, and compares each checkpoint with the starting point on the same 120-round development suite using matched lineups and seeds. The final holdout is run once for only the selected checkpoint.

> The native `main.py` fixes all four agents for the lifetime of a process, while the replay buffer is not persisted across processes. This experiment therefore uses 20-round microbatches: every 200-round stage contains 4×A, 3×B, 2×C, and 1×D, with both the microbatch order and the three-opponent order shuffled. The framework independently randomizes starting positions each round. This preserves the exact 40/30/20/10 mixture without restarting every round and preventing replay warm-up.

Safety constraint: this notebook never overwrites `agent_code/dueling_hunt_center_04`. All writes go to a new working agent, `experiments/checkpoints/`, `experiments/configs/`, `experiments/records/`, or `results/ipynb08_continue_dddqn/`.

## 1. Imports and run switches

Run only the preflight first. After confirming its output, enable `PREPARE`, `RUN_TRAINING`, and `RUN_DEV_EVAL` in that order. `RUN_FINAL_HOLDOUT` must remain `False` until exactly one candidate has been selected.

In [33]:
from __future__ import annotations

import hashlib
import itertools
import json
import os
import random
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

def find_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / 'main.py').is_file() and (directory / 'agent_code').is_dir():
            return directory
    raise FileNotFoundError('Could not find the project root containing main.py and agent_code/')

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

PYTHON = Path(sys.executable)
PREPARE = True
RUN_TRAINING = False
RUN_DEV_EVAL = False
RUN_FINAL_HOLDOUT = True  # one use only, after selecting one checkpoint
print('root:', ROOT)
print('python:', PYTHON)

root: <PROJECT_ROOT>
python: <PYTHON_EXECUTABLE>


## 2. Frozen protocol

The fixed RNG chooses the 20-round type-D block from two permitted variants: `rule + day7 × 2` or `rule + dueling_ddqn × 2`. All DDQN opponents run in evaluation mode; only the working agent in the first list position is enabled by `--train 1`.

In [27]:
EXPERIMENT = 'ipynb08_continue_training_dddqn'
BASE_AGENT = 'dueling_hunt_center_04'
TRAIN_AGENT = 'dueling_hunt_center_04_mix_train'
TRAIN_SEED = 1701
STAGE_SIZE = 200
MICROBATCH_SIZE = 20
STAGE_ENDS = (200, 400, 600, 800, 1000)

AGENT_ROOT = ROOT / 'agent_code'
CHECKPOINT_DIR = ROOT / 'experiments/checkpoints'
CONFIG_DIR = ROOT / 'experiments/configs'
RECORD_DIR = ROOT / 'experiments/records'
RESULT_DIR = ROOT / 'results' / EXPERIMENT
WORKING_DIR = AGENT_ROOT / TRAIN_AGENT
MODEL_NAME = 'dueling_ddqn_model.pt'

LINEUPS = {
    'A': ('rule_based_agent',) * 3,
    'B': ('rule_based_agent', 'rule_based_agent', 'frozen_ddqn_day7'),
    'C': ('rule_based_agent', 'frozen_ddqn_day7', 'dueling_ddqn_agent'),
}
D_OPTIONS = (
    ('rule_based_agent', 'frozen_ddqn_day7', 'frozen_ddqn_day7'),
    ('rule_based_agent', 'dueling_ddqn_agent', 'dueling_ddqn_agent'),
)
BLOCK_TYPES = tuple('AAAABBBCCD')  # 10 × 20 = 200 rounds

DEV_LINEUPS = {
    'rule3': ('rule_based_agent',) * 3,
    'rule2_day7': ('rule_based_agent', 'rule_based_agent', 'frozen_ddqn_day7'),
    'rule_day7_dddqn': ('rule_based_agent', 'frozen_ddqn_day7', 'dueling_ddqn_agent'),
    'rule_day7x2': ('rule_based_agent', 'frozen_ddqn_day7', 'frozen_ddqn_day7'),
    'rule_dddqnx2': ('rule_based_agent', 'dueling_ddqn_agent', 'dueling_ddqn_agent'),
    'day7_dddqnx2': ('frozen_ddqn_day7', 'dueling_ddqn_agent', 'dueling_ddqn_agent'),
}
DEV_SEEDS = (8101, 8102, 8103, 8104, 8105, 8101)
FINAL_SEEDS = (18101, 18102, 18103, 18104, 18105, 18106)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

protocol = {
    'experiment': EXPERIMENT,
    'base_agent': BASE_AGENT,
    'working_agent': TRAIN_AGENT,
    'training_seed': TRAIN_SEED,
    'rounds': 1000,
    'checkpoint_every': STAGE_SIZE,
    'microbatch_rounds': MICROBATCH_SIZE,
    'mixture': {'A': 0.40, 'B': 0.30, 'C': 0.20, 'D': 0.10},
    'lineups': {**{k: list(v) for k, v in LINEUPS.items()}, 'D_options': [list(x) for x in D_OPTIONS]},
    'dev_seeds': list(DEV_SEEDS),
    'final_holdout_seeds': list(FINAL_SEEDS),
    'selection_rule': {
        'target_paired_mean_score_gain': 0.3,
        'coins_must_not_decrease': True,
        'max_relative_suicide_increase': 0.20,
        'invalid_must_not_increase': True,
        'mean_action_ms_limit': 500.0,
    },
}
protocol

{'experiment': 'ipynb08_continue_training_dddqn',
 'base_agent': 'dueling_hunt_center_04',
 'working_agent': 'dueling_hunt_center_04_mix_train',
 'training_seed': 1701,
 'rounds': 1000,
 'checkpoint_every': 200,
 'microbatch_rounds': 20,
 'mixture': {'A': 0.4, 'B': 0.3, 'C': 0.2, 'D': 0.1},
 'lineups': {'A': ['rule_based_agent', 'rule_based_agent', 'rule_based_agent'],
  'B': ['rule_based_agent', 'rule_based_agent', 'frozen_ddqn_day7'],
  'C': ['rule_based_agent', 'frozen_ddqn_day7', 'dueling_ddqn_agent'],
  'D_options': [['rule_based_agent', 'frozen_ddqn_day7', 'frozen_ddqn_day7'],
   ['rule_based_agent', 'dueling_ddqn_agent', 'dueling_ddqn_agent']]},
 'dev_seeds': [8101, 8102, 8103, 8104, 8105, 8101],
 'final_holdout_seeds': [18101, 18102, 18103, 18104, 18105, 18106],
 'selection_rule': {'target_paired_mean_score_gain': 0.3,
  'coins_must_not_decrease': True,
  'max_relative_suicide_increase': 0.2,
  'invalid_must_not_increase': True,
  'mean_action_ms_limit': 500.0}}

## 3. Preflight and isolated working copy

`dueling_hunt_center_04` is an inference-only directory. Preparation copies its callbacks, features, model definition, and checkpoint, then copies `train.py`, `replay.py`, and `symmetry.py` from the current `dueling_ddqn_agent` implementation of the same training algorithm. This step is idempotent: an existing working directory, configuration set, and 0000 checkpoint are safely reused when their contents match. Missing files or content drift produce explicit errors; nothing is silently repaired or overwritten.

In [28]:
required_agents = [BASE_AGENT, 'rule_based_agent', 'frozen_ddqn_day7', 'dueling_ddqn_agent']
for name in required_agents:
    assert (AGENT_ROOT / name / 'callbacks.py').exists(), f'missing agent: {name}'
assert (AGENT_ROOT / BASE_AGENT / MODEL_NAME).exists(), 'missing base checkpoint'
for filename in ('train.py', 'replay.py', 'symmetry.py'):
    assert (AGENT_ROOT / 'dueling_ddqn_agent' / filename).exists(), filename
assert (AGENT_ROOT / BASE_AGENT / 'features.py').read_bytes() == (AGENT_ROOT / 'dueling_ddqn_agent' / 'features.py').read_bytes(), 'feature code drift'
assert 'HUNT_CENTER_Q_BIAS = 0.04' in (AGENT_ROOT / BASE_AGENT / 'callbacks.py').read_text(), 'wrong hunt-center policy'
subprocess.run([str(PYTHON), '-c', 'import torch, numpy; print(torch.__version__)'], cwd=ROOT, check=True)
import torch
base_checkpoint = torch.load(AGENT_ROOT / BASE_AGENT / MODEL_NAME, map_location='cpu', weights_only=True)
assert base_checkpoint.get('optimizer_state') is not None, 'optimizer state would be reset'
assert int(base_checkpoint.get('environment_steps', 0)) > 0, 'exploration schedule state is missing'
assert int(base_checkpoint.get('learner_updates', 0)) > 0, 'learner update state is missing'
print('base sha256:', sha256(AGENT_ROOT / BASE_AGENT / MODEL_NAME))
print('preflight OK')

def _assert_same_file(actual: Path, expected: Path):
    assert actual.exists(), f'Incomplete working copy; missing {actual}'
    assert actual.read_bytes() == expected.read_bytes(), f'Working code drift: {actual.name}'

def _write_json_once(path: Path, payload):
    if path.exists():
        assert json.loads(path.read_text()) == payload, f'Protocol drift: {path}'
        return
    path.write_text(json.dumps(payload, indent=2) + '\n')

def prepare_working_copy():
    reused_existing = WORKING_DIR.exists()
    if reused_existing:
        # Reuse a completed preparation (or a later trained model), but never repair or overwrite silently.
        for filename in ('callbacks.py', 'features.py', 'model.py'):
            _assert_same_file(WORKING_DIR / filename, AGENT_ROOT / BASE_AGENT / filename)
        for filename in ('train.py', 'replay.py', 'symmetry.py'):
            _assert_same_file(WORKING_DIR / filename, AGENT_ROOT / 'dueling_ddqn_agent' / filename)
        assert (WORKING_DIR / MODEL_NAME).exists(), 'Working checkpoint is missing'
    else:
        WORKING_DIR.mkdir(parents=True)
        for filename in ('callbacks.py', 'features.py', 'model.py', MODEL_NAME):
            shutil.copy2(AGENT_ROOT / BASE_AGENT / filename, WORKING_DIR / filename)
        for filename in ('train.py', 'replay.py', 'symmetry.py'):
            shutil.copy2(AGENT_ROOT / 'dueling_ddqn_agent' / filename, WORKING_DIR / filename)
        (WORKING_DIR / 'README.md').write_text(
            '# Working continuation agent\n\nCreated by notebooks/08_continue_training_dddqn.ipynb. '
            'Starts from dueling_hunt_center_04 and preserves HUNT_CENTER_Q_BIAS=0.04.\n'
        )
    for directory in (CHECKPOINT_DIR, CONFIG_DIR, RECORD_DIR, RESULT_DIR):
        directory.mkdir(parents=True, exist_ok=True)
    start = CHECKPOINT_DIR / 'hunt_center_04_mix_0000.pt'
    if not start.exists():
        checkpoint = torch.load(AGENT_ROOT / BASE_AGENT / MODEL_NAME, map_location='cpu', weights_only=True)
        checkpoint['continuation'] = {
            'completed_rounds': 0, 'base_agent': BASE_AGENT,
            'base_sha256': sha256(AGENT_ROOT / BASE_AGENT / MODEL_NAME),
            'protocol': protocol, 'stage_schedule': [],
            'saved_utc': datetime.now(timezone.utc).isoformat(),
        }
        torch.save(checkpoint, start)
    start_checkpoint = torch.load(start, map_location='cpu', weights_only=True)
    assert start_checkpoint['continuation']['completed_rounds'] == 0
    assert start_checkpoint['continuation']['base_sha256'] == sha256(AGENT_ROOT / BASE_AGENT / MODEL_NAME)
    _write_json_once(CONFIG_DIR / f'{EXPERIMENT}.json', protocol)
    dev_spec = {'split': 'development', 'enabled': False, 'rounds_per_lineup': 20,
                'seeds': list(DEV_SEEDS), 'lineups': {k: list(v) for k, v in DEV_LINEUPS.items()},
                'candidates': [BASE_AGENT] + [f'hunt_center_04_mix_{x:04d}' for x in STAGE_ENDS]}
    final_spec = {'split': 'final_holdout', 'enabled': False, 'run_once': True, 'candidate': None,
                  'rounds_per_lineup': 20, 'seeds': list(FINAL_SEEDS),
                  'lineups': {k: list(v) for k, v in DEV_LINEUPS.items()}}
    _write_json_once(CONFIG_DIR / f'{EXPERIMENT}_dev.json', dev_spec)
    _write_json_once(CONFIG_DIR / f'{EXPERIMENT}_final_holdout.json', final_spec)
    return {'working_agent': str(WORKING_DIR.relative_to(ROOT)),
            'start_checkpoint': str(start.relative_to(ROOT)),
            'reused_existing': reused_existing}

if PREPARE:
    print(prepare_working_copy())
else:
    print('Dry run only: set PREPARE=True once to create the isolated working copy.')

2.14.0
base sha256: 709cca4351f3bc5893f766b01b8c9335fa299d25331a275ae281b123df31009a
preflight OK
{'working_agent': 'agent_code/dueling_hunt_center_04_mix_train', 'start_checkpoint': 'experiments/checkpoints/hunt_center_04_mix_0000.pt', 'reused_existing': True}


## 4. Generate and execute the 1000-round schedule

Each command trains only the first agent in the list; all opponents remain frozen. Every block has its own recorded environment seed. At the end of each stage, the checkpoint is augmented with cumulative training rounds, the complete protocol, the actual stage order, seeds, starting-checkpoint hash, and timestamp. It is then copied into an inference-only agent directory for consistent evaluation. Cell 09 is idempotent: completed stages are reused only after validating the manifest, statistics, checkpoint metadata and hash, frozen code, and model. Any incomplete or drifted artifact produces an explicit error.

In [29]:
def make_schedule():
    rng = random.Random(TRAIN_SEED)
    schedule = []
    for stage_index, stage_end in enumerate(STAGE_ENDS, start=1):
        block_types = list(BLOCK_TYPES)
        rng.shuffle(block_types)
        for block_index, kind in enumerate(block_types, start=1):
            opponents = D_OPTIONS[rng.randrange(len(D_OPTIONS))] if kind == 'D' else LINEUPS[kind]
            opponents = list(opponents)
            rng.shuffle(opponents)
            schedule.append({
                'stage': stage_index, 'stage_end': stage_end, 'block': block_index,
                'type': kind, 'rounds': MICROBATCH_SIZE, 'opponents': opponents,
                'seed': TRAIN_SEED + stage_index * 100 + block_index,
            })
    return schedule

SCHEDULE = make_schedule()
assert len(SCHEDULE) * MICROBATCH_SIZE == 1000
for stage in range(1, 6):
    counts = {kind: sum(x['rounds'] for x in SCHEDULE if x['stage'] == stage and x['type'] == kind) for kind in 'ABCD'}
    assert counts == {'A': 80, 'B': 60, 'C': 40, 'D': 20}
    print('stage', stage, counts, [x['type'] for x in SCHEDULE if x['stage'] == stage])

def _validate_stats_file(stats_path, expected_agent, expected_rounds):
    assert stats_path.exists(), f'Missing result file: {stats_path}'
    payload = json.loads(stats_path.read_text())
    assert len(payload.get('by_round', {})) == expected_rounds, f'Wrong round count: {stats_path}'
    names = [name for name in payload.get('by_agent', {})
             if name == expected_agent or name.startswith(expected_agent + '_')]
    assert len(names) == 1, f'Expected exactly one {expected_agent} in {stats_path}; found {names}'
    recorded_rounds = int(payload['by_agent'][names[0]].get('rounds', 0))
    assert recorded_rounds == expected_rounds, f'Agent round count mismatch: {stats_path}'
    return payload

def run_command(command, *, stats_path, expected_agent, expected_rounds, run_metadata):
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    sidecar = stats_path.with_suffix('.run.json')
    if stats_path.exists():
        _validate_stats_file(stats_path, expected_agent, expected_rounds)
        assert sidecar.exists(), f'Result exists without provenance sidecar: {sidecar}'
        expected_sidecar = {**run_metadata, 'stats_sha256': sha256(stats_path)}
        assert json.loads(sidecar.read_text()) == expected_sidecar, f'Run protocol/result drift: {sidecar}'
        print('REUSE result:', stats_path.relative_to(ROOT))
        return False
    assert not sidecar.exists(), f'Orphan provenance sidecar without result: {sidecar}'
    subprocess.run(command, cwd=ROOT, check=True)
    _validate_stats_file(stats_path, expected_agent, expected_rounds)
    sidecar.write_text(json.dumps({**run_metadata, 'stats_sha256': sha256(stats_path)}, indent=2) + '\n')
    return True

def _assert_training_code_current():
    assert WORKING_DIR.exists(), 'Working agent is missing; run preparation first'
    for filename in ('callbacks.py', 'features.py', 'model.py'):
        _assert_same_file(WORKING_DIR / filename, AGENT_ROOT / BASE_AGENT / filename)
    for filename in ('train.py', 'replay.py', 'symmetry.py'):
        _assert_same_file(WORKING_DIR / filename, AGENT_ROOT / 'dueling_ddqn_agent' / filename)
    assert (WORKING_DIR / MODEL_NAME).exists(), 'Working checkpoint is missing'

def _assert_immutable_protocol_assets():
    import torch
    expected_configs = {
        CONFIG_DIR / f'{EXPERIMENT}.json': protocol,
        CONFIG_DIR / f'{EXPERIMENT}_dev.json': {
            'split': 'development', 'enabled': False, 'rounds_per_lineup': 20,
            'seeds': list(DEV_SEEDS), 'lineups': {k: list(v) for k, v in DEV_LINEUPS.items()},
            'candidates': [BASE_AGENT] + [f'hunt_center_04_mix_{x:04d}' for x in STAGE_ENDS],
        },
        CONFIG_DIR / f'{EXPERIMENT}_final_holdout.json': {
            'split': 'final_holdout', 'enabled': False, 'run_once': True, 'candidate': None,
            'rounds_per_lineup': 20, 'seeds': list(FINAL_SEEDS),
            'lineups': {k: list(v) for k, v in DEV_LINEUPS.items()},
        },
    }
    for path, expected in expected_configs.items():
        assert path.exists(), f'Missing immutable config: {path}'
        assert json.loads(path.read_text()) == expected, f'Config protocol drift: {path}'
    start = CHECKPOINT_DIR / 'hunt_center_04_mix_0000.pt'
    assert start.exists(), f'Missing immutable starting checkpoint: {start}'
    initial = torch.load(start, map_location='cpu', weights_only=True)
    source = torch.load(AGENT_ROOT / BASE_AGENT / MODEL_NAME, map_location='cpu', weights_only=True)
    continuation = initial.get('continuation', {})
    assert continuation.get('completed_rounds') == 0, '0000 checkpoint round metadata drift'
    assert continuation.get('protocol') == protocol, '0000 checkpoint protocol drift'
    assert continuation.get('base_sha256') == sha256(AGENT_ROOT / BASE_AGENT / MODEL_NAME), '0000 base hash drift'
    assert initial.get('environment_steps') == source.get('environment_steps'), '0000 environment steps drift'
    assert initial.get('learner_updates') == source.get('learner_updates'), '0000 learner updates drift'
    assert initial['policy_state'].keys() == source['policy_state'].keys()
    assert all(torch.equal(initial['policy_state'][key], source['policy_state'][key])
               for key in source['policy_state']), '0000 policy weights differ from frozen base'

def _validate_stage_artifacts(stage_end, stage_rows, manifest_row=None):
    import torch
    destination = CHECKPOINT_DIR / f'hunt_center_04_mix_{stage_end:04d}.pt'
    frozen_name = f'hunt_center_04_mix_{stage_end:04d}'
    frozen_dir = AGENT_ROOT / frozen_name
    assert destination.exists(), f'Missing stage checkpoint: {destination}'
    assert frozen_dir.is_dir(), f'Missing frozen agent directory: {frozen_dir}'
    checkpoint = torch.load(destination, map_location='cpu', weights_only=True)
    assert checkpoint.get('optimizer_state') is not None, f'Missing optimizer state: {destination}'
    assert 'policy_state' in checkpoint, f'Missing policy state: {destination}'
    continuation = checkpoint.get('continuation', {})
    assert continuation.get('completed_rounds') == stage_end, f'Checkpoint round metadata drift: {destination}'
    assert continuation.get('base_agent') == BASE_AGENT, f'Checkpoint base-agent drift: {destination}'
    assert continuation.get('base_sha256') == sha256(AGENT_ROOT / BASE_AGENT / MODEL_NAME), f'Base hash drift: {destination}'
    assert continuation.get('protocol') == protocol, f'Checkpoint protocol drift: {destination}'
    assert continuation.get('stage_schedule') == stage_rows, f'Checkpoint schedule drift: {destination}'
    for filename in ('callbacks.py', 'features.py', 'model.py'):
        _assert_same_file(frozen_dir / filename, WORKING_DIR / filename)
    assert sha256(frozen_dir / MODEL_NAME) == sha256(destination), f'Frozen model differs from checkpoint: {frozen_name}'
    if manifest_row is not None:
        assert manifest_row.get('stage') == stage_end // STAGE_SIZE
        assert manifest_row.get('completed_rounds') == stage_end
        assert manifest_row.get('schedule') == stage_rows
        assert manifest_row.get('checkpoint') == str(destination.relative_to(ROOT))
        assert manifest_row.get('checkpoint_sha256') == sha256(destination), f'Manifest hash drift: {destination}'
        assert manifest_row.get('frozen_agent') == frozen_name
    return destination, frozen_name

def _checkpoint_value_equal(left, right):
    import torch
    if isinstance(left, torch.Tensor) and isinstance(right, torch.Tensor):
        return torch.equal(left, right)
    if isinstance(left, dict) and isinstance(right, dict):
        return left.keys() == right.keys() and all(_checkpoint_value_equal(left[key], right[key]) for key in left)
    if isinstance(left, (list, tuple)) and isinstance(right, (list, tuple)):
        return len(left) == len(right) and all(_checkpoint_value_equal(a, b) for a, b in zip(left, right))
    return left == right

def freeze_stage(stage_end, stage_rows):
    import torch
    active = WORKING_DIR / MODEL_NAME
    destination = CHECKPOINT_DIR / f'hunt_center_04_mix_{stage_end:04d}.pt'
    frozen_name = f'hunt_center_04_mix_{stage_end:04d}'
    frozen_dir = AGENT_ROOT / frozen_name
    if destination.exists() or frozen_dir.exists():
        assert destination.exists() and frozen_dir.is_dir(), f'Incomplete stage artifacts for {stage_end}'
        print('REUSE checkpoint:', destination.relative_to(ROOT))
        return _validate_stage_artifacts(stage_end, stage_rows)
    checkpoint = torch.load(active, map_location='cpu', weights_only=True)
    assert 'policy_state' in checkpoint and 'optimizer_state' in checkpoint
    checkpoint['continuation'] = {
        'completed_rounds': stage_end,
        'base_agent': BASE_AGENT,
        'base_sha256': sha256(AGENT_ROOT / BASE_AGENT / MODEL_NAME),
        'protocol': protocol,
        'stage_schedule': stage_rows,
        'saved_utc': datetime.now(timezone.utc).isoformat(),
    }
    temporary = destination.with_suffix('.pt.tmp')
    torch.save(checkpoint, temporary)
    temporary.replace(destination)

    frozen_dir.mkdir()
    for filename in ('callbacks.py', 'features.py', 'model.py'):
        shutil.copy2(WORKING_DIR / filename, frozen_dir / filename)
    shutil.copy2(destination, frozen_dir / MODEL_NAME)
    (frozen_dir / 'README.md').write_text(f'# Frozen checkpoint after {stage_end} continuation rounds\n')
    return _validate_stage_artifacts(stage_end, stage_rows)

def train_all_stages():
    _assert_immutable_protocol_assets()
    _assert_training_code_current()
    manifest_path = RECORD_DIR / f'{EXPERIMENT}_training.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        assert manifest.get('experiment') == EXPERIMENT, 'Training manifest experiment drift'
        assert manifest.get('protocol') == protocol, 'Training manifest protocol drift'
    else:
        existing = list((RESULT_DIR / 'training').glob('*.json')) + list(CHECKPOINT_DIR.glob('hunt_center_04_mix_0[2-9]00.pt'))
        assert not existing, 'Training artifacts exist but the training manifest is missing'
        manifest = {'experiment': EXPERIMENT, 'protocol': protocol, 'stages': []}
    completed_rows = {row['completed_rounds']: row for row in manifest.get('stages', [])}
    assert list(completed_rows) == list(STAGE_ENDS[:len(completed_rows)]), 'Training manifest stages are not a valid prefix'
    for stage, stage_end in enumerate(STAGE_ENDS, start=1):
        rows = [row for row in SCHEDULE if row['stage'] == stage]
        if stage_end in completed_rows:
            for row in rows:
                stats = RESULT_DIR / 'training' / f"stage{stage}_block{row['block']:02d}_{row['type']}.json"
                _validate_stats_file(stats, TRAIN_AGENT, row['rounds'])
            _validate_stage_artifacts(stage_end, rows, completed_rows[stage_end])
            print('REUSE completed stage:', stage_end)
            continue
        for row in rows:
            stats = RESULT_DIR / 'training' / f"stage{stage}_block{row['block']:02d}_{row['type']}.json"
            command = [str(PYTHON), 'main.py', 'play', '--no-gui', '--train', '1',
                       '--agents', TRAIN_AGENT, *row['opponents'], '--scenario', 'classic',
                       '--n-rounds', str(row['rounds']), '--seed', str(row['seed']),
                       '--save-stats', str(stats)]
            print('RUN', ' '.join(map(str, command[1:])))
            run_metadata = {'experiment': EXPERIMENT, 'split': 'training', 'stage': stage,
                            'schedule_row': row, 'command': list(map(str, command[1:])), 'protocol': protocol}
            run_command(command, stats_path=stats, expected_agent=TRAIN_AGENT,
                        expected_rounds=row['rounds'], run_metadata=run_metadata)
        checkpoint, frozen_name = freeze_stage(stage_end, rows)
        manifest['stages'].append({
            'stage': stage, 'completed_rounds': stage_end, 'schedule': rows,
            'checkpoint': str(checkpoint.relative_to(ROOT)),
            'checkpoint_sha256': sha256(checkpoint), 'frozen_agent': frozen_name,
        })
        completed_rows[stage_end] = manifest['stages'][-1]
        temporary_manifest = manifest_path.with_suffix('.json.tmp')
        temporary_manifest.write_text(json.dumps(manifest, indent=2) + '\n')
        temporary_manifest.replace(manifest_path)
    if len(completed_rows) == len(STAGE_ENDS):
        final_checkpoint = CHECKPOINT_DIR / f'hunt_center_04_mix_{STAGE_ENDS[-1]:04d}.pt'
        active_state = torch.load(WORKING_DIR / MODEL_NAME, map_location='cpu', weights_only=True)
        frozen_state = torch.load(final_checkpoint, map_location='cpu', weights_only=True)
        for key in ('policy_state', 'optimizer_state', 'environment_steps', 'learner_updates'):
            assert _checkpoint_value_equal(active_state.get(key), frozen_state.get(key)), f'Working/final training state differs: {key}'
    return manifest

if RUN_TRAINING:
    training_manifest = train_all_stages()
else:
    print('Training disabled. Inspect SCHEDULE, then set RUN_TRAINING=True.')

stage 1 {'A': 80, 'B': 60, 'C': 40, 'D': 20} ['C', 'A', 'B', 'A', 'A', 'B', 'A', 'D', 'B', 'C']
stage 2 {'A': 80, 'B': 60, 'C': 40, 'D': 20} ['A', 'B', 'C', 'D', 'A', 'B', 'A', 'B', 'A', 'C']
stage 3 {'A': 80, 'B': 60, 'C': 40, 'D': 20} ['A', 'C', 'B', 'A', 'D', 'A', 'C', 'B', 'B', 'A']
stage 4 {'A': 80, 'B': 60, 'C': 40, 'D': 20} ['B', 'A', 'C', 'B', 'D', 'C', 'A', 'B', 'A', 'A']
stage 5 {'A': 80, 'B': 60, 'C': 40, 'D': 20} ['A', 'A', 'C', 'D', 'A', 'B', 'C', 'B', 'A', 'B']
REUSE completed stage: 200
REUSE completed stage: 400
REUSE completed stage: 600
REUSE completed stage: 800
REUSE completed stage: 1000


## 5. Paired 120-round development validation

All six checkpoints (0/200/400/600/800/1000) use exactly the same six lineups, 20-round counts, and seeds. Every candidate occupies the first position in the agent list; the framework reproduces maps and starting positions from the same seed. The primary statistic is the per-round score difference from the starting checkpoint, with a 95% confidence interval from 10,000 bootstrap resamples. Cell 11 is idempotent: a complete manifest and matching raw statistics are validated and reused without replaying matches. Missing files, candidate/lineup/seed/command drift, or a model-hash mismatch stops execution.

In [30]:
CANDIDATES = {0: BASE_AGENT, **{end: f'hunt_center_04_mix_{end:04d}' for end in STAGE_ENDS}}

def _validate_candidate_artifact(rounds_done, candidate):
    candidate_dir = AGENT_ROOT / candidate
    assert candidate_dir.is_dir(), f'Missing candidate directory: {candidate_dir}'
    for filename in ('callbacks.py', 'features.py', 'model.py'):
        _assert_same_file(candidate_dir / filename, AGENT_ROOT / BASE_AGENT / filename)
    model = candidate_dir / MODEL_NAME
    assert model.exists(), f'Missing candidate model: {model}'
    if rounds_done == 0:
        assert sha256(model) == sha256(AGENT_ROOT / BASE_AGENT / MODEL_NAME), 'Baseline model drift'
    else:
        training_manifest_path = RECORD_DIR / f'{EXPERIMENT}_training.json'
        assert training_manifest_path.exists(), 'Missing training manifest for continuation candidate'
        training_manifest = json.loads(training_manifest_path.read_text())
        assert training_manifest.get('protocol') == protocol, 'Training protocol drift'
        matches = [row for row in training_manifest.get('stages', []) if row.get('completed_rounds') == rounds_done]
        assert len(matches) == 1, f'Missing or duplicate training stage {rounds_done}'
        checkpoint = ROOT / matches[0]['checkpoint']
        assert checkpoint.exists(), f'Missing candidate checkpoint: {checkpoint}'
        assert matches[0]['checkpoint_sha256'] == sha256(checkpoint), f'Checkpoint hash drift: {checkpoint}'
        assert sha256(model) == sha256(checkpoint), f'Candidate model differs from checkpoint: {candidate}'

def _expected_evaluation_entry(rounds_done, candidate, *, holdout=False):
    seeds = FINAL_SEEDS if holdout else DEV_SEEDS
    split = 'final_holdout' if holdout else 'dev'
    rows = []
    for index, ((lineup_name, opponents), seed) in enumerate(zip(DEV_LINEUPS.items(), seeds), start=1):
        stats = RESULT_DIR / split / candidate / f'{index:02d}_{lineup_name}.json'
        relative_stats = stats.relative_to(ROOT)
        command = [str(PYTHON), 'main.py', 'play', '--no-gui', '--agents', candidate, *opponents,
                   '--scenario', 'classic', '--n-rounds', '20', '--seed', str(seed),
                   '--save-stats', str(relative_stats)]
        rows.append({'lineup': lineup_name, 'seed': seed, 'stats': str(stats.relative_to(ROOT)),
                     'command': list(map(str, command[1:]))})
    return {'rounds_done': rounds_done, 'candidate': candidate, 'split': split, 'runs': rows}

def _validate_evaluation_entry(actual, expected):
    assert actual == expected, f"Evaluation manifest drift: {expected['candidate']} / {expected['split']}"
    _validate_candidate_artifact(expected['rounds_done'], expected['candidate'])
    for row in expected['runs']:
        _validate_stats_file(ROOT / row['stats'], expected['candidate'], 20)

def evaluate_candidate(rounds_done, candidate, *, holdout=False):
    _assert_immutable_protocol_assets()
    _validate_candidate_artifact(rounds_done, candidate)
    expected = _expected_evaluation_entry(rounds_done, candidate, holdout=holdout)
    for row in expected['runs']:
        stats = ROOT / row['stats']
        command = [str(PYTHON), *row['command']]
        run_metadata = {'experiment': EXPERIMENT, 'split': expected['split'],
                        'rounds_done': rounds_done, 'candidate': candidate,
                        'lineup': row['lineup'], 'seed': row['seed'],
                        'command': row['command'], 'protocol': protocol}
        print('CHECK', ' '.join(map(str, row['command'])))
        run_command(command, stats_path=stats, expected_agent=candidate, expected_rounds=20,
                    run_metadata=run_metadata)
    return expected

if RUN_DEV_EVAL:
    _assert_immutable_protocol_assets()
    manifest_path = RECORD_DIR / f'{EXPERIMENT}_dev_manifest.json'
    expected_manifest = {'experiment': EXPERIMENT, 'candidates': [
        _expected_evaluation_entry(rounds_done, candidate)
        for rounds_done, candidate in CANDIDATES.items()
    ]}
    if manifest_path.exists():
        dev_manifest = json.loads(manifest_path.read_text())
        assert dev_manifest == expected_manifest, f'Development manifest protocol drift: {manifest_path}'
        for actual, expected in zip(dev_manifest['candidates'], expected_manifest['candidates']):
            _validate_evaluation_entry(actual, expected)
        print('REUSE complete development validation:', manifest_path.relative_to(ROOT))
    else:
        dev_manifest = {'experiment': EXPERIMENT, 'candidates': []}
        for rounds_done, candidate in CANDIDATES.items():
            dev_manifest['candidates'].append(evaluate_candidate(rounds_done, candidate))
        assert dev_manifest == expected_manifest
        temporary_manifest = manifest_path.with_suffix('.json.tmp')
        temporary_manifest.write_text(json.dumps(dev_manifest, indent=2) + '\n')
        temporary_manifest.replace(manifest_path)
else:
    print('Development evaluation disabled.')

REUSE complete development validation: experiments/records/ipynb08_continue_training_dddqn_dev_manifest.json


## 6. Aggregate, bootstrap, and selection report

The selection logic does not automatically choose the 1,000-round version. It first excludes candidates with think time ≥500 ms or failed safety guardrails, then selects the eligible checkpoint with the highest mean score. If no continued checkpoint passes the guardrails, the starting checkpoint is retained. The report marks a plateau after two consecutive checkpoints without a clear improvement.

In [31]:
from evaluation.summarize_stats import summarize

def stats_paths(candidate, split='dev'):
    return sorted((RESULT_DIR / split / candidate).glob('*.json'))

def per_round_scores(paths, candidate):
    values = []
    for path in paths:
        payload = json.loads(path.read_text())
        for round_data in payload.get('by_round', {}).values():
            agents = round_data['agents']
            names = [name for name in agents if name == candidate or name.startswith(candidate + '_')]
            assert len(names) == 1, (path, names)
            values.append(float(agents[names[0]].get('score', 0.0)))
    return np.asarray(values, dtype=float)

def paired_ci(base, candidate, seed=20260915, draws=10_000):
    assert base.shape == candidate.shape and base.size == 120
    differences = candidate - base
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, differences.size, size=(draws, differences.size))
    boot = differences[indices].mean(axis=1)
    return float(differences.mean()), [float(x) for x in np.quantile(boot, [0.025, 0.975])]

def build_dev_report():
    base_name = CANDIDATES[0]
    base_paths = stats_paths(base_name)
    assert len(base_paths) == 6, 'Run the full development suite first'
    base_scores = per_round_scores(base_paths, base_name)
    base_metrics = summarize(base_paths, base_name)
    rows = []
    no_clear_gain = 0
    plateau_at = None
    previous_score = base_metrics['mean_score']
    for rounds_done, candidate in CANDIDATES.items():
        paths = stats_paths(candidate)
        assert len(paths) == 6, candidate
        metrics = summarize(paths, candidate)
        gain, ci = paired_ci(base_scores, per_round_scores(paths, candidate))
        suicide_limit = base_metrics['suicides_per_round'] * 1.20
        if base_metrics['suicides_per_round'] == 0:
            suicide_limit = 0.0
        safe = (metrics['coins_per_round'] >= base_metrics['coins_per_round'] and
                metrics['suicides_per_round'] <= suicide_limit and
                metrics['invalid_per_round'] <= base_metrics['invalid_per_round'] and
                metrics['mean_action_ms'] < 500.0)
        clear_vs_previous = rounds_done == 0 or metrics['mean_score'] >= previous_score + 0.3
        if rounds_done:
            no_clear_gain = 0 if clear_vs_previous else no_clear_gain + 1
            if no_clear_gain >= 2 and plateau_at is None:
                plateau_at = rounds_done
        rows.append({**metrics, 'rounds_done': rounds_done, 'paired_score_gain_vs_0000': gain,
                     'paired_score_gain_ci95': ci, 'gain_target_met': gain >= 0.3,
                     'safety_guardrails_pass': safe, 'clear_gain_vs_previous': clear_vs_previous})
        previous_score = metrics['mean_score']
    eligible = [row for row in rows if row['rounds_done'] == 0 or row['safety_guardrails_pass']]
    selected = max(eligible, key=lambda row: (row['mean_score'], -row['suicides_per_round'], -row['rounds_done']))
    return {
        'experiment': EXPERIMENT, 'created_utc': datetime.now(timezone.utc).isoformat(),
        'protocol': protocol, 'checkpoints': rows, 'plateau_detected_at_round': plateau_at,
        'selected_rounds': selected['rounds_done'], 'selected_agent': CANDIDATES[selected['rounds_done']],
        'final_holdout_has_run': False,
    }

dev_report = None
if RUN_DEV_EVAL:
    dev_report = build_dev_report()
    report_path = RECORD_DIR / f'{EXPERIMENT}_report.json'
    report_path.write_text(json.dumps(dev_report, indent=2) + '\n')
    print(json.dumps(dev_report, indent=2))
else:
    print('Aggregation waits for RUN_DEV_EVAL=True and all 36 validation commands.')

{
  "experiment": "ipynb08_continue_training_dddqn",
  "created_utc": "2026-09-15T11:58:08.331534+00:00",
  "protocol": {
    "experiment": "ipynb08_continue_training_dddqn",
    "base_agent": "dueling_hunt_center_04",
    "working_agent": "dueling_hunt_center_04_mix_train",
    "training_seed": 1701,
    "rounds": 1000,
    "checkpoint_every": 200,
    "microbatch_rounds": 20,
    "mixture": {
      "A": 0.4,
      "B": 0.3,
      "C": 0.2,
      "D": 0.1
    },
    "lineups": {
      "A": [
        "rule_based_agent",
        "rule_based_agent",
        "rule_based_agent"
      ],
      "B": [
        "rule_based_agent",
        "rule_based_agent",
        "frozen_ddqn_day7"
      ],
      "C": [
        "rule_based_agent",
        "frozen_ddqn_day7",
        "dueling_ddqn_agent"
      ],
      "D_options": [
        [
          "rule_based_agent",
          "frozen_ddqn_day7",
          "frozen_ddqn_day7"
        ],
        [
          "rule_based_agent",
          "dueling_ddqn_age

## Development validation result and interpretation

Under the prespecified selection rules, no continuation checkpoint can be confirmed as better than the original model, so `dueling_hunt_center_04` should remain the submission candidate. However, `0200` and `0800` both show promising improvement signals, with `0800` providing the more balanced performance across opponent lineups.

| checkpoint | mean score | gain vs 0000 | paired 95% CI | coins | kills | suicides | invalid |
|---|---:|---:|---:|---:|---:|---:|---:|
| original 0000 | 3.300 | — | — | 2.467 | 0.167 | 0.267 | 2.550 |
| 0200 | **3.883** | **+0.583** | [-0.075, 1.258] | 2.675 | **0.242** | 0.117 | 5.008 |
| 0400 | 3.692 | +0.392 | [-0.242, 1.025] | **2.775** | 0.183 | 0.125 | 4.933 |
| 0600 | 3.617 | +0.317 | [-0.308, 0.959] | 2.533 | 0.217 | 0.125 | 4.767 |
| 0800 | 3.867 | +0.567 | [-0.050, 1.192] | 2.742 | 0.225 | 0.150 | **4.442** |
| 1000 | 3.583 | +0.283 | [-0.350, 0.917] | 2.750 | 0.167 | **0.083** | 5.275 |

Key findings:

- `0200` has the highest overall mean score and simultaneously improves crates, kills, suicides, survival, and win rate.
- `0800` scores only 0.017 below `0200` and exceeds the baseline in five of six lineups, compared with four of six for `0200`; `0800` is therefore more balanced across opponents.
- The paired bootstrap 95% confidence interval crosses zero for every continuation checkpoint, so the current 120-round development suite is insufficient to establish a statistically clear improvement.
- Every continuation checkpoint violates the invalid-action guardrail: the baseline records 2.55 invalid actions per round versus 4.44–5.28 for the continued models. After normalization, the rate still rises from 7.75 to approximately 13.7–15.7 invalid actions per 1,000 steps, so longer survival alone does not explain the increase.
- `1000` scores below both `0200` and `0800` and misses the +0.3 target gain. The development report detects a plateau at 600 rounds, so neither additional training nor automatic selection of the final checkpoint is recommended.
- All checkpoint parameters are finite and all optimizer states are present. Learner updates increase from 68,516 to 135,370, confirming that the network weights were genuinely updated rather than merely copied. Every version runs at approximately 2.5 ms/action, safely below the 500 ms limit.

Selection recommendation: retain the original `dueling_hunt_center_04` for the formal submission. If one continuation checkpoint is admitted to the one-shot final holdout, choose `hunt_center_04_mix_0800`: it nearly matches the highest mean score, is more stable across lineups, and has the lowest invalid-action rate among the continuation checkpoints. Do not change the selected checkpoint after inspecting the final holdout.

## 7. One-shot final holdout

Run this section only after selecting exactly one candidate from the development suite. Restart the kernel, keep `RUN_TRAINING=False` and `RUN_DEV_EVAL=False`, enter `SELECTED_AGENT` manually, and then set `RUN_FINAL_HOLDOUT=True`. Do not repeatedly switch checkpoints based on final-holdout results.

In [34]:
SELECTED_AGENT = "hunt_center_04_mix_0800"

if RUN_FINAL_HOLDOUT:
    assert SELECTED_AGENT in CANDIDATES.values(), 'Select exactly one frozen development candidate'
    marker = RECORD_DIR / f'{EXPERIMENT}_FINAL_HOLDOUT_USED.json'
    if marker.exists():
        raise RuntimeError(f'Final holdout was already used: {marker}')
    rounds_done = next(k for k, v in CANDIDATES.items() if v == SELECTED_AGENT)
    final_manifest = evaluate_candidate(rounds_done, SELECTED_AGENT, holdout=True)
    final_paths = stats_paths(SELECTED_AGENT, split='final_holdout')
    final_metrics = summarize(final_paths, SELECTED_AGENT)
    payload = {**final_manifest, 'metrics': final_metrics, 'used_utc': datetime.now(timezone.utc).isoformat()}
    marker.write_text(json.dumps(payload, indent=2) + '\n')
    print(json.dumps(payload, indent=2))
else:
    print('Final holdout locked (expected during training/model selection).')

CHECK main.py play --no-gui --agents hunt_center_04_mix_0800 rule_based_agent rule_based_agent rule_based_agent --scenario classic --n-rounds 20 --seed 18101 --save-stats <PROJECT_ROOT>/results/ipynb08_continue_training_dddqn/final_holdout/hunt_center_04_mix_0800/01_rule3.json


100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


CHECK main.py play --no-gui --agents hunt_center_04_mix_0800 rule_based_agent rule_based_agent frozen_ddqn_day7 --scenario classic --n-rounds 20 --seed 18102 --save-stats <PROJECT_ROOT>/results/ipynb08_continue_training_dddqn/final_holdout/hunt_center_04_mix_0800/02_rule2_day7.json


100%|██████████| 20/20 [00:37<00:00,  1.88s/it]


CHECK main.py play --no-gui --agents hunt_center_04_mix_0800 rule_based_agent frozen_ddqn_day7 dueling_ddqn_agent --scenario classic --n-rounds 20 --seed 18103 --save-stats <PROJECT_ROOT>/results/ipynb08_continue_training_dddqn/final_holdout/hunt_center_04_mix_0800/03_rule_day7_dddqn.json


...

CHECK main.py play --no-gui --agents hunt_center_04_mix_0800 rule_based_agent dueling_ddqn_agent dueling_ddqn_agent --scenario classic --n-rounds 20 --seed 18105 --save-stats <PROJECT_ROOT>/results/ipynb08_continue_training_dddqn/final_holdout/hunt_center_04_mix_0800/05_rule_dddqnx2.json


100%|██████████| 20/20 [00:53<00:00,  2.68s/it]


CHECK main.py play --no-gui --agents hunt_center_04_mix_0800 frozen_ddqn_day7 dueling_ddqn_agent dueling_ddqn_agent --scenario classic --n-rounds 20 --seed 18106 --save-stats <PROJECT_ROOT>/results/ipynb08_continue_training_dddqn/final_holdout/hunt_center_04_mix_0800/06_day7_dddqnx2.json


100%|██████████| 20/20 [01:08<00:00,  3.42s/it]


{
  "rounds_done": 800,
  "candidate": "hunt_center_04_mix_0800",
  "split": "final_holdout",
  "runs": [
    {
      "lineup": "rule3",
      "seed": 18101,
      "stats": "results/ipynb08_continue_training_dddqn/final_holdout/hunt_center_04_mix_0800/01_rule3.json",
      "command": [
        "main.py",
        "play",
        "--no-gui",
        "--agents",
        "hunt_center_04_mix_0800",
        "rule_based_agent",
        "rule_based_agent",
        "rule_based_agent",
        "--scenario",
        "classic",
        "--n-rounds",
        "20",
        "--seed",
        "18101",
        "--save-stats",
        "<PROJECT_ROOT>/results/ipynb08_continue_training_dddqn/final_holdout/hunt_center_04_mix_0800/01_rule3.json"
      ]
    },
    {
      "lineup": "rule2_day7",
      "seed": 18102,
      "stats": "results/ipynb08_continue_training_dddqn/final_holdout/hunt_center_04_mix_0800/02_rule2_day7.json",
      "command": [
        "main.py",
        "play",
        "--no-gui",
     

## Final holdout result and submission decision

The one-shot final holdout was completed on 2026-09-15 for the preselected `hunt_center_04_mix_0800`: six lineups with 20 rounds each, for 120 rounds total, using seeds 18101–18106 that were not used for training or development selection. The marker was written to `experiments/records/ipynb08_continue_training_dddqn_FINAL_HOLDOUT_USED.json`; the holdout must not be rerun, and a different continuation checkpoint must not be selected in response to these results.

### Aggregate result

| metric | 0800 development | 0800 final holdout | holdout − dev |
|---|---:|---:|---:|
| mean score | 3.867 | 3.583 | -0.283 |
| win rate | 0.417 | 0.308 | -0.108 |
| mean rank | 2.000 | 2.208 | +0.208 (worse) |
| score margin | 0.794 | 0.569 | -0.225 |
| coins / round | 2.742 | 2.750 | +0.008 |
| crates / round | 30.958 | 31.675 | +0.717 |
| kills / round | 0.225 | 0.167 | -0.058 |
| suicides / round | 0.150 | 0.150 | 0.000 |
| invalid / round | 4.442 | 5.150 | +0.708 |
| survival rate | 0.692 | 0.758 | +0.067 |
| mean action time | 2.543 ms | 2.530 ms | -0.013 ms |

### Holdout breakdown by opponent lineup

| lineup | score | win rate | score margin | kills | suicides | invalid |
|---|---:|---:|---:|---:|---:|---:|
| rule × 3 | **5.45** | **0.55** | **+2.767** | 0.30 | 0.10 | 3.40 |
| rule × 2 + day7 | 4.30 | 0.30 | +1.400 | 0.30 | 0.20 | 5.90 |
| rule + day7 + DDQN | 3.25 | 0.35 | +0.750 | 0.05 | 0.10 | 6.40 |
| rule + day7 × 2 | 3.45 | 0.25 | -0.150 | 0.30 | 0.25 | 6.10 |
| rule + DDQN × 2 | **2.30** | 0.20 | **-1.017** | 0.00 | 0.15 | 4.40 |
| day7 + DDQN × 2 | 2.75 | 0.20 | -0.333 | 0.05 | 0.10 | 4.70 |

### Interpretation

The final holdout confirms that the 0800 weights retain resource collection and survival: coins are unchanged, crates and survival improve, suicide rate remains 0.15, and inference stays safely below the 500 ms limit. However, the tournament-facing metrics regress relative to the same checkpoint's development result: score, win rate, rank, margin, and kills all worsen, while invalid actions rise further. The lineup split shows a clear asymmetry: the agent is strong against `rule_based_agent × 3`, but loses average score margin and produces few kills in the two DDQN-heavy lineups.

This final holdout evaluated only the preselected 0800 candidate, not the original model on the same holdout seeds. It therefore cannot provide a new paired 0800-versus-baseline estimate. Nevertheless, the evidence does not overturn the development decision: the development CI for 0800 crossed zero and every continuation checkpoint failed the invalid-action guardrail; the independent holdout then showed weaker tournament outcomes rather than a clean replication of the development gain.

**Final decision:** do not promote `hunt_center_04_mix_0800` as the tournament submission weight. Retain `dueling_hunt_center_04` as the submission candidate, archive `0800` as a promising but unconfirmed experimental checkpoint, and do not consume the final holdout again.

## 8. Report checklist

Before submission, confirm the following:

- The SHA-256 of the original `dueling_hunt_center_04/dueling_ddqn_model.pt` is unchanged.
- All six checkpoints at 0/200/400/600/800/1000 rounds contain the policy, optimizer, environment steps, learner updates, and continuation metadata.
- `experiments/records/ipynb08_continue_training_dddqn_training.json` records all 50 executed training microbatches.
- The development report includes mean score, coins, crates, kills, suicides, invalid actions, think time, and the paired 95% confidence interval.
- Final selection is based on the best safe checkpoint rather than defaulting to the 1,000-round version.
- The final-holdout marker is created only once, and the final report documents any regression, plateau, or decision to revert to the 0000 checkpoint.